## What this project is about

We compare four tabular reinforcement learning algorithms to see how each one performs under different environment conditions. All four are implemented from scratch with no RL libraries.

The four algorithms:
1. Q-learning (Watkins and Dayan 1992) - off policy, learns optimal policy
2. SARSA (Rummery and Niranjan 1994) - on policy, learns safer policy near hazards
3. Expected SARSA (Van Seijen et al 2009) - lower variance version of SARSA
4. Double Q-learning (Van Hasselt 2010) - fixes overestimation in Q-learning

All algorithms store Q values in a 2D table called a Q-table of shape (n_states, n_actions). Each cell Q[s, a] stores the estimated future reward for taking action a in state s. The best action in any state is just argmax over the row.

## Experiments (my work)

- **Experiment 2**: Compare all four algorithms across 7 environments with 30 seeds
- **Experiment 3**: Sweep hyperparameters (alpha, gamma, epsilon decay) to find sensitivities  
- **Experiment 4**: Measure Q-value overestimation to show maximization bias in Q-learning

Experiment 1 (convergence theory) is Khyati's contribution.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.algorithms import QLearning, SARSA, ExpectedSARSA, DoubleQLearning, value_iteration
from src.environments import make_env, ENV_CONFIGS
from src.utils import (
    run_experiment, run_sensitivity_sweep, run_sensitivity_2d, run_bias_experiment,
    plot_convergence, plot_sensitivity_heatmap, plot_sensitivity_1d,
    plot_bias_comparison, plot_policy_grid, plot_q_rmse,
    plot_final_performance_table, compute_policy_optimality,
    statistical_comparison, train_agent, compute_confidence_interval, smooth,
    ALGO_COLORS
)

sns.set_theme(style='whitegrid', font_scale=1.1)
print('All imports successful.')
print(f'Available environments: {list(ENV_CONFIGS.keys())}')

## 1. Algorithm Implementations

All four algorithms update a Q-table after every step. The difference is what target they use.

| Algorithm | Update Target | What makes it different |
|-----------|--------------|------------------------|
| Q-learning | r + gamma * max Q(s', a') | uses the best possible next Q value even if the agent did not take that action |
| SARSA | r + gamma * Q(s', a') where a' is actually taken | uses the Q value of whatever action the agent will actually do next |
| Expected SARSA | r + gamma * weighted average Q(s') | averages over all possible next actions weighted by their probability under the policy |
| Double Q-learning | r + gamma * Q2(s', argmax Q1(s')) | Q1 picks the action, Q2 evaluates it to remove overestimation |

Key terms:
- **alpha**: learning rate, how much to move Q values per update (0.1 means shift 10% toward target)
- **gamma**: discount factor, how much future rewards matter vs immediate (0.99 is far-sighted)
- **epsilon**: probability of picking a random action instead of the best known one
- **TD error**: target minus current estimate, the correction signal applied each step
- **on policy**: learns the value of the policy it is actually following including random exploration
- **off policy**: always learns toward the optimal greedy policy regardless of what the agent actually does

### 1.1 Sanity Check on Custom GridWorld

Before running the full experiments we test all four algorithms on our hand-built 4x4 grid. This environment is small enough that value iteration gives us the exact correct Q* in milliseconds. If each algorithm's learned Q-table is close to Q* and its policy matches the optimal policy, we know the code is correct.

We check three things: policy optimality (what percent of states have the right action), Q-RMSE (how far the learned Q values are from Q*), and mean reward over the last 100 episodes.

In [ ]:
# Quick sanity check: all algorithms should converge on the simple Custom GridWorld
env, _ = make_env('CustomGrid-4x4')
q_star, v_star, opt_policy = value_iteration(env, gamma=0.99)

print('Value Iteration Results (Custom GridWorld):')
print(f'Optimal policy: {opt_policy}')
print(f'V* (reshaped 4x4):\n{v_star.reshape(4,4).round(3)}\n')

# Test each algorithm
for name, AgentClass in [('Q-learning', QLearning), ('SARSA', SARSA),
                          ('Expected SARSA', ExpectedSARSA), ('Double Q-learning', DoubleQLearning)]:
    np.random.seed(42)
    env, _ = make_env('CustomGrid-4x4')
    agent = AgentClass(n_states=16, n_actions=4, alpha=0.1, gamma=0.99,
                       epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995)
    rewards = train_agent(env, agent, n_episodes=2000, max_steps=100)
    learned_policy = agent.get_policy()
    optimality = compute_policy_optimality(agent.get_q_table(), opt_policy)
    rmse = np.sqrt(np.mean((agent.get_q_table() - q_star) ** 2))
    print(f'{name:20s} | Policy optimality: {optimality:5.1f}% | Q-RMSE: {rmse:.4f} | Mean reward (last 100): {np.mean(rewards[-100:]):.3f}')

## 2. Experiment 2: Four-Algorithm Comparison

We run all four algorithms on seven environments with 30 random seeds each. Using 30 seeds means results are statistically reliable and not just due to a lucky random initialization.

Environments tested:

| Environment | States | Stochastic | Main challenge |
|-------------|--------|-----------|----------------|
| CustomGrid-4x4 | 16 | No | Baseline, correctness check |
| FrozenLake-4x4-det | 16 | No | Simple deterministic grid |
| FrozenLake-4x4-slip | 16 | Yes | Each move only works 1 out of 3 times |
| FrozenLake-8x8-det | 64 | No | Larger state space |
| FrozenLake-8x8-slip | 64 | Yes | Large + stochastic |
| CliffWalking | 48 | No | Shows on policy vs off policy difference clearly |
| Taxi-v3 | 500 | No | Tests if tabular methods scale |

All four algorithms use the same hyperparameters within each environment so any difference in performance is due to the algorithm itself, not tuning advantages.

In [ ]:
ALGOS = ['Q-learning', 'SARSA', 'Expected SARSA', 'Double Q-learning']
N_SEEDS = 30

EXP2_CONFIGS = {
    'CustomGrid-4x4':      {'n_episodes': 2000, 'max_steps': 100,  'alpha': 0.1,  'gamma': 0.99, 'epsilon_decay': 0.995},
    'FrozenLake-4x4-det':  {'n_episodes': 2000, 'max_steps': 100,  'alpha': 0.1,  'gamma': 0.99, 'epsilon_decay': 0.995},
    'FrozenLake-4x4-slip': {'n_episodes': 5000, 'max_steps': 100,  'alpha': 0.05, 'gamma': 0.99, 'epsilon_decay': 0.999},
    'FrozenLake-8x8-det':  {'n_episodes': 5000, 'max_steps': 200,  'alpha': 0.1,  'gamma': 0.99, 'epsilon_decay': 0.998},
    'FrozenLake-8x8-slip': {'n_episodes': 10000,'max_steps': 200,  'alpha': 0.05, 'gamma': 0.99, 'epsilon_decay': 0.9995},
    'CliffWalking':        {'n_episodes': 1000, 'max_steps': 200,  'alpha': 0.1,  'gamma': 0.99, 'epsilon_decay': 0.995},
    'Taxi':                {'n_episodes': 3000, 'max_steps': 200,  'alpha': 0.1,  'gamma': 0.99, 'epsilon_decay': 0.998},
}

print(f"Running {len(ALGOS)} algorithms on {len(EXP2_CONFIGS)} environments with {N_SEEDS} seeds each.")
print(f"Total training runs: {len(ALGOS) * len(EXP2_CONFIGS) * N_SEEDS:,}")
total_eps = sum(cfg['n_episodes'] for cfg in EXP2_CONFIGS.values()) * len(ALGOS) * N_SEEDS
print(f"Total episodes to train: {total_eps:,}")

In [ ]:
all_results = {}   # {env_name: {algo_name: (n_seeds, n_episodes) array}}
all_q_tables = {}  # {env_name: {algo_name: list of Q-tables}}
all_q_stars = {}   # {env_name: Q* array or None}

for env_name, cfg in EXP2_CONFIGS.items():
    print(f'\nRunning {env_name}...')
    results, q_tables, q_star = run_experiment(
        env_name, ALGOS,
        n_episodes=cfg['n_episodes'],
        n_seeds=N_SEEDS,
        max_steps=cfg['max_steps'],
        alpha=cfg['alpha'],
        gamma=cfg['gamma'],
        epsilon_decay=cfg['epsilon_decay'],
    )
    all_results[env_name] = results
    all_q_tables[env_name] = q_tables
    all_q_stars[env_name] = q_star
    print(f'  Done. Shapes: {[(k, v.shape) for k,v in results.items()]}')

print('\nAll Experiment 2 runs complete.')

### 2.1 Convergence Curves

Two panels per environment. Left shows smoothed reward per episode with a shaded 95% confidence band across the 30 seeds. Right shows cumulative reward which rewards algorithms that learn faster since they collect more reward earlier. A curve rising quickly and leveling off means good convergence. Wide CI bands mean inconsistent results across seeds.

In [ ]:
# Plot convergence for each environment
for env_name, results in all_results.items():
    window = 100 if 'slip' in env_name or '8x8' in env_name else 50
    plot_convergence(results, env_name, window=window,
                     save_path=f'../results/convergence_{env_name}.png')

### 2.2 Q-value RMSE vs Q*

After training we compare each algorithm's final Q-table against Q* from value iteration. RMSE = sqrt(mean((Q(s,a) minus Q*(s,a))^2)) over all state action pairs. Lower is better. An algorithm can get the right policy (correct best action per state) while still having high RMSE if its Q values are off numerically. RMSE measures how well calibrated the values are, not just whether the ranking is correct.

In [ ]:
# Q-value RMSE for environments where Q* is available
for env_name in all_results:
    q_star = all_q_stars[env_name]
    if q_star is not None:
        plot_q_rmse(all_q_tables[env_name], q_star, env_name,
                    save_path=f'../results/q_rmse_{env_name}.png')

### 2.3 Policy Visualization on CliffWalking

CliffWalking is the classic example from Sutton and Barto that shows the difference between on policy and off policy learning. The bottom row is a cliff that ends the episode with a large negative reward. Q-learning learns to walk right along the cliff edge because it evaluates the optimal greedy policy. SARSA learns to take the safer route one row above because it evaluates the policy including random exploration, and knows it might accidentally step off the cliff during training. Each cell in the grid shows an arrow for the greedy action in that state. Red cells are cliff, green is goal.

In [ ]:
# CliffWalking: show Q-learning (cliff-edge optimal) vs SARSA (safe path)
cliff_q = all_q_tables.get('CliffWalking', {})
if cliff_q:
    # CliffWalking is 4 rows x 12 cols = 48 states
    cliff_states = set(range(37, 47))  # cliff cells
    goal_states = {47}  # bottom-right

    for algo_name in ['Q-learning', 'SARSA']:
        if algo_name in cliff_q:
            # Use the first seed's Q-table for visualization
            plot_policy_grid(cliff_q[algo_name][0], 'CliffWalking',
                           nrow=4, ncol=12,
                           title=f'{algo_name} Policy (CliffWalking)',
                           hole_states=cliff_states,
                           goal_states=goal_states,
                           save_path=f'../results/policy_cliff_{algo_name.replace(" ","_")}.png')

### 2.4 Statistical Comparison

We use Welch's t-test (does not assume equal variance between groups) to check if performance differences between algorithm pairs are statistically significant or just noise. The output shows mean reward, 95% confidence interval, and standard deviation for each algorithm, then a pairwise table with t statistic, p value, and Cohen's d.

Cohen's d is the effect size: 0.2 is small, 0.5 is medium, 0.8 is large. Significance stars: *** means p < 0.001, ** means p < 0.01, * means p < 0.05, blank means not significant.

In [ ]:
# Statistical comparison for each environment
for env_name, results in all_results.items():
    statistical_comparison(results, env_name)

### 2.5 Final Performance Summary

Bar chart comparing all four algorithms across all seven environments. Each bar is the mean reward over the last 100 episodes averaged across all 30 seeds. Error bars are 95% confidence intervals.

In [ ]:
# Summary bar chart across all environments
plot_final_performance_table(all_results, save_path='../results/final_performance_summary.png')

### 2.6 Policy Optimality Rate

For each state we check if the agent's best learned action (argmax of its Q-table row) matches the optimal action from value iteration. Policy optimality is the percentage of states where these agree. An algorithm can have 100% policy optimality even if the actual Q values are wrong, as long as the best action per state is ranked correctly.

In [ ]:
# Policy optimality: % states matching optimal policy
print(f"{'Environment':<25} {'Algorithm':<20} {'Policy Optimality %':>20}")
print('=' * 67)
for env_name in all_results:
    q_star = all_q_stars[env_name]
    if q_star is not None:
        opt_policy = np.argmax(q_star, axis=1)
        for algo_name in ALGOS:
            q_list = all_q_tables[env_name][algo_name]
            opt_rates = [compute_policy_optimality(q, opt_policy) for q in q_list]
            mean_opt = np.mean(opt_rates)
            std_opt = np.std(opt_rates)
            print(f'{env_name:<25} {algo_name:<20} {mean_opt:>17.1f}% +/- {std_opt:.1f}')

## 3. Experiment 3: Hyperparameter Sensitivity Analysis

The original papers prove convergence but do not give practical guidance on how to set alpha, gamma, and epsilon. This experiment fills that gap by systematically sweeping each parameter and recording the effect on final performance.

Parameters swept:
- **alpha**: 0.01, 0.05, 0.1, 0.2, 0.5 - too small means slow learning, too large means instability
- **gamma**: 0.8, 0.9, 0.95, 0.99 - controls how much future rewards matter
- **epsilon decay**: 0.99, 0.995, 0.999, 0.9999 - controls how fast exploration drops off

We test on FrozenLake-4x4-det, FrozenLake-4x4-slip, and CliffWalking with 10 seeds per configuration. Performance is measured as mean reward over the last 100 episodes.

In [ ]:
# Sensitivity sweep parameters
ALPHA_VALUES = [0.01, 0.05, 0.1, 0.2, 0.5]
GAMMA_VALUES = [0.8, 0.9, 0.95, 0.99]
EPSILON_DECAY_VALUES = [0.99, 0.995, 0.999, 0.9999]

# Environments for sensitivity analysis
SENS_ENVS = ['FrozenLake-4x4-det', 'FrozenLake-4x4-slip', 'CliffWalking']
SENS_SEEDS = 10  # fewer seeds for computational efficiency

### 3.1 Alpha x Gamma Heatmaps

Each heatmap shows how the combination of learning rate (rows) and discount factor (columns) affects one algorithm on one environment. Green means high performance, red means low. A broad green region means the algorithm is robust to those parameters. A narrow green strip means it needs careful tuning. We generate 12 heatmaps total (4 algorithms x 3 environments).

In [ ]:
# 2D heatmap: alpha vs gamma for each algorithm on each environment
for env_name in SENS_ENVS:
    n_eps = EXP2_CONFIGS[env_name]['n_episodes']
    max_s = EXP2_CONFIGS[env_name]['max_steps']
    
    for algo_name in ALGOS:
        print(f'  Sweeping alpha x gamma: {algo_name} on {env_name}...')
        grid = run_sensitivity_2d(
            env_name, algo_name,
            'alpha', ALPHA_VALUES,
            'gamma', GAMMA_VALUES,
            n_episodes=n_eps, n_seeds=SENS_SEEDS, max_steps=max_s,
        )
        plot_sensitivity_heatmap(
            grid, 'alpha', ALPHA_VALUES, 'gamma', GAMMA_VALUES,
            algo_name, env_name,
            save_path=f'../results/sens_alpha_gamma_{algo_name.replace(" ","_")}_{env_name}.png'
        )

### 3.2 Learning Rate Sensitivity

Shows how alpha affects final performance for all four algorithms on the same axes. A flat line means the algorithm works well regardless of learning rate. A peaked curve means there is a sweet spot and performance drops off on either side. Stochastic environments tend to be more sensitive to alpha because noisy updates need a smaller step size to stay stable.

In [ ]:
# 1D alpha sweep: all algorithms on same plot
for env_name in SENS_ENVS:
    n_eps = EXP2_CONFIGS[env_name]['n_episodes']
    max_s = EXP2_CONFIGS[env_name]['max_steps']
    
    alpha_results = {}
    for algo_name in ALGOS:
        means, stds = run_sensitivity_sweep(
            env_name, algo_name, 'alpha', ALPHA_VALUES,
            n_episodes=n_eps, n_seeds=SENS_SEEDS, max_steps=max_s,
        )
        alpha_results[algo_name] = (means, stds)
    
    plot_sensitivity_1d(alpha_results, 'alpha', ALPHA_VALUES, env_name,
                        save_path=f'../results/sens_alpha_{env_name}.png')

### 3.3 Epsilon Decay Sensitivity

Epsilon decay controls how fast the agent switches from random exploration to using its learned policy. With decay 0.99, epsilon reaches near zero by episode 460. With 0.9999 it still has 37% exploration at episode 10000. Too fast and the agent commits to a bad policy before exploring enough. Too slow and it wastes episodes exploring randomly when it already knows a good policy. Deterministic environments can handle faster decay since the agent gets consistent feedback each time it tries an action.

In [ ]:
# Epsilon decay sweep
for env_name in SENS_ENVS:
    n_eps = EXP2_CONFIGS[env_name]['n_episodes']
    max_s = EXP2_CONFIGS[env_name]['max_steps']
    
    decay_results = {}
    for algo_name in ALGOS:
        means, stds = run_sensitivity_sweep(
            env_name, algo_name, 'epsilon_decay', EPSILON_DECAY_VALUES,
            n_episodes=n_eps, n_seeds=SENS_SEEDS, max_steps=max_s,
        )
        decay_results[algo_name] = (means, stds)
    
    plot_sensitivity_1d(decay_results, 'epsilon_decay', EPSILON_DECAY_VALUES, env_name,
                        save_path=f'../results/sens_eps_decay_{env_name}.png')

## 4. Experiment 4: Maximization Bias

Q-learning always uses the max Q value from the next state as its update target. When Q values are noisy estimates (which they always are early in training), the maximum of a set of noisy numbers is always higher than the true maximum. This is called maximization bias and it causes Q-learning to be overconfident about the value of certain states.

Double Q-learning fixes this by keeping two tables. Q1 picks which action looks best, Q2 evaluates how good that action actually is. Since the two tables are trained on different samples their errors are independent and the upward bias cancels out.

We measure overestimation as: mean over all states of (max Q(s) minus max Q*(s)). Positive means overestimation. We use fixed epsilon throughout so exploration stays constant and does not confound the signal.

In [ ]:
# Run bias experiment on FrozenLake slippery (stochastic → more bias)
BIAS_ENVS = ['FrozenLake-4x4-slip', 'FrozenLake-4x4-det', 'CliffWalking']

for env_name in BIAS_ENVS:
    print(f'Running bias experiment on {env_name}...')
    cfg = EXP2_CONFIGS[env_name]
    q_overest, dq_overest, q_star = run_bias_experiment(
        env_name,
        n_episodes=cfg['n_episodes'],
        n_seeds=N_SEEDS,
        alpha=cfg['alpha'],
        gamma=0.99,
        epsilon=0.1,
        epsilon_min=0.1,
        epsilon_decay=1.0,  # fixed epsilon for clearer bias signal
        max_steps=cfg['max_steps'],
    )
    plot_bias_comparison(q_overest, dq_overest, env_name,
                         save_path=f'../results/bias_{env_name}.png')
    
    # Print final overestimation stats
    q_final = np.mean(q_overest[:, -100:], axis=1)
    dq_final = np.mean(dq_overest[:, -100:], axis=1)
    print(f'  Q-learning  overest: {np.mean(q_final):.4f} +/- {np.std(q_final):.4f}')
    print(f'  Double Q-learning:   {np.mean(dq_final):.4f} +/- {np.std(dq_final):.4f}')
    t_stat, p_val = stats.ttest_ind(q_final, dq_final)
    print(f'  t={t_stat:.3f}, p={p_val:.6f} {"***" if p_val<0.001 else "**" if p_val<0.01 else "*" if p_val<0.05 else "ns"}')

## 5. Key Findings and Discussion

### 5.1 Summary Table

Mean reward over the last 100 episodes across all environments. For FrozenLake, 1.0 means the agent always reaches the goal and 0.0 means it never does. For CliffWalking, values around -13 are optimal and more negative means the agent is falling off cliffs or taking longer paths. For Taxi, positive values mean successful deliveries.

In [ ]:
# Summary table
print('\n' + '='*90)
print('SUMMARY: Mean Reward (last 100 episodes) across all environments')
print('='*90)
header = f'{"Environment":<25}'
for algo in ALGOS:
    header += f'{algo:>16}'
print(header)
print('-'*90)

for env_name, results in all_results.items():
    row = f'{env_name:<25}'
    for algo in ALGOS:
        data = results[algo]
        final = np.mean(data[:, -100:])  # mean over seeds and last 100 eps
        row += f'{final:>16.4f}'
    print(row)

In [ ]:
# Create comprehensive multi-panel figure
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Select key environments for the summary figure
summary_envs = ['FrozenLake-4x4-det', 'FrozenLake-4x4-slip', 'CliffWalking',
                'FrozenLake-8x8-det', 'Taxi', 'CustomGrid-4x4']

for idx, env_name in enumerate(summary_envs):
    if env_name not in all_results:
        continue
    ax = axes[idx // 3, idx % 3]
    results = all_results[env_name]
    window = 100 if 'slip' in env_name or '8x8' in env_name else 50
    
    for algo_name, data in results.items():
        mean, ci = compute_confidence_interval(data)
        sm_mean = smooth(mean, window)
        x = np.arange(len(sm_mean))
        color = ALGO_COLORS.get(algo_name, 'gray')
        ax.plot(x, sm_mean, label=algo_name, color=color, linewidth=1.2)
    
    ax.set_title(env_name, fontsize=11)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    if idx == 0:
        ax.legend(fontsize=8, frameon=True)

plt.suptitle('TD Control Algorithm Comparison Across Environments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/summary_all_envs.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Key Takeaways

**Deterministic environments**: All four algorithms converge to similar performance on the simple CustomGrid with no statistically significant differences. On FrozenLake-4x4-det, Double Q-learning underperforms significantly (p < 0.01, Cohen d = 0.78) because its two table mechanism halves the effective update rate when there is no bias to correct.

**Stochastic environments**: Q-learning gets the highest success rate on FrozenLake-4x4-slip. Double Q-learning struggles most in stochastic settings because slower convergence from two tables is not worth the bias correction benefit in small environments.

**FrozenLake 8x8**: All algorithms get near zero reward. The state space is too large and reward too sparse for tabular methods to converge in the given training budget. This finding itself is important since it shows the limits of tabular RL.

**CliffWalking**: Q-learning achieves better reward (-16.7 vs -17.6 for SARSA, p = 0.018). This matches the textbook prediction that off policy methods learn the optimal cliff-edge path while on policy SARSA learns the safer detour.

**Taxi-v3**: Q-learning dominates at 500 states. Double Q-learning fails to converge at all in the allotted budget because each of its two tables only gets updated half the time, which is not enough for a 500 state environment.

**Maximization bias**: On CliffWalking, Q-learning shows clear overestimation (+1.90 above Q*) while Double Q-learning stays much closer to zero. The difference is significant (t = 19.0, p < 0.001), confirming Van Hasselt 2010.

**Sensitivity**: Performance is most sensitive to learning rate in stochastic environments. High gamma (0.99) works well but requires more careful alpha tuning. Deterministic environments tolerate faster epsilon decay.

## References

1. Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning*, 8, 279–292.
2. Rummery, G. A., & Niranjan, M. (1994). On-line Q-learning using connectionist systems. CUED/F-INFENG/TR 166.
3. Van Seijen, H., et al. (2009). Expected SARSA. *IEEE Symposium on ADPRL*, 177–184.
4. Van Hasselt, H. (2010). Double Q-learning. *NIPS*, 23.
5. Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning* (2nd ed.). MIT Press.